# Telco Customer Churn: End-to-End Data Analysis Project

Project directory structure:

```
project_root/
├── data/raw/Telco-Customer-Churn.csv   # raw data
├── src/telco_churn/cleaning.py         # reusable cleaning functions
├── notebooks/this_file.ipynb
├── requirements.txt
└── README.md
```

**Business Problem**: Which customers are likely to churn? The output can inform retention budgets and customer service prioritization (offline modeling demo, not a production pipeline).

## Workflow Overview

| Stage | Description |
|-------|-------------|
| Load | Read CSV from `data/raw` |
| Clean | Modular function `clean_telco_churn` |
| EDA | Structure, target distribution, visualizations, churn rate by segment |
| Preprocessing | One-hot encoding, stratified train/test split |
| Modeling | Logistic Regression, Random Forest, HistGradientBoosting |
| Evaluation | ROC-AUC, classification_report |

> **Purpose**: Align the reproducible project structure with the problem statement.
> **Why**: GitHub/learning projects should let readers know where files come from and where code runs — avoid notebooks that float without data or modules.


## 0. Environment and Data Paths

The kernel can be launched from the project root or `notebooks/` — the code below searches upward for the project root containing `data/raw` and adds `src` to `sys.path` so the cleaning module can be imported.


In [ ]:
import sys


> **Purpose**: Portable path resolution and project module import.
> **Why**: A common issue in teaching repos is that running from `notebooks/` breaks `src` imports; searching for the project root reduces these errors.


## 1. Load Raw Data

Using `pandas.read_csv`; if encoding issues arise, try `encoding='utf-8-sig'`.


In [ ]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()


> **Purpose**: Load the file referenced in the README.
> **Why**: Downstream metrics must be traceable to a filename and row count — this is the first step of a data contract.


## 2. Data Cleaning

Rules are implemented in `src/telco_churn/cleaning.py` (shared with this notebook). Key steps: strip column names and strings, deduplicate on primary key, convert `TotalCharges` to numeric and fill missing values, validate categories and numeric ranges.


In [ ]:
df = clean_telco_churn(df)

print("\nMissing values after cleaning (should be none or handled):")
miss = df.isna().sum()
print(miss[miss > 0] if miss.sum() else "No remaining missing values")


> **Purpose**: Ensure one row per customer with correct data types.
> **Why**: Duplicate IDs distort churn rates; `TotalCharges` as string breaks models; new customers with zero tenure missing total charges requires a business-aligned fill value.


## 3. Exploratory Data Analysis (EDA)

Inspect data types, descriptive statistics, target distribution, and the relationship between MonthlyCharges and Churn on the cleaned `df`.


In [ ]:
df.info()
df.describe(include="all")


In [ ]:
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("\nChurn ratio:")
print(df["Churn"].value_counts(normalize=True))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["Churn"].value_counts().plot(kind="bar", ax=axes[0], title="Churn count")
df.boxplot(column="MonthlyCharges", by="Churn", ax=axes[1])
plt.suptitle("")
axes[1].set_title("MonthlyCharges vs Churn")
plt.tight_layout()
plt.show()


> **Purpose**: Confirm class imbalance and monthly charge distribution differences.
> **Why**: Class proportions affect metric interpretation; charts help form testable hypotheses.


### 3.1 Business Insight: Churn Rate by Segment

Translate data into operational language (contract, payment, plan, tenure). When results align with model feature directions, cross-team trust is easier to build. **Correlation does not imply causation.**


In [ ]:
def churn_rate_by(col: str) -> pd.Series:
    return df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)

print("Overall churn rate:", f"{(df['Churn'] == 'Yes').mean():.1%}")
print("\nContract:")
print(churn_rate_by("Contract"))
print("\nPaymentMethod:")
print(churn_rate_by("PaymentMethod"))
print("\nInternetService:")
print(churn_rate_by("InternetService"))

df["_tb"] = pd.cut(
    df["tenure"],
    bins=[-1, 0, 12, 24, 60, 1000],
    labels=["0", "1-12", "13-24", "25-60", ">60"],
)
print("\nTenure bin:")
print(df.groupby("_tb", observed=True)["Churn"].apply(lambda s: (s == "Yes").mean()))
df.drop(columns=["_tb"], inplace=True)

p75 = df["TotalCharges"].quantile(0.75)
hq_m2m = (df["Contract"] == "Month-to-month") & (df["TotalCharges"] >= p75)
print("\nHigh cumulative (P75+) month-to-month — Churn rate:", f"{(df.loc[hq_m2m, 'Churn'] == 'Yes').mean():.1%}", "n=", int(hq_m2m.sum()))

> **Purpose**: Align with common telecom narratives (month-to-month, electronic check, Fiber, tenure).
> **Why**: Models need to explain to business teams "why this list"; grouped rates are the most intuitive reference.


## 4. High-Value Customer Analysis

Identify where high-value customers (by monthly charge × tenure) are churning.


In [ ]:
# High-value customers: MonthlyCharges > median AND tenure > 12 months
median_charge = df["MonthlyCharges"].median()
high_value = df[(df["MonthlyCharges"] > median_charge) & (df["tenure"] > 12)].copy()

total_customers = len(df)
hv_count = len(high_value)
hv_churn = (high_value["Churn"] == "Yes").mean()
overall_churn = (df["Churn"] == "Yes").mean()

print(f"Total customers: {total_customers:,}")
print(f"High-value customers: {hv_count:,} ({hv_count/total_customers:.1%} of total)")
print(f"Overall churn rate: {overall_churn:.1%}")
print(f"High-value customer churn rate: {hv_churn:.1%}")
direction = 'higher' if hv_churn > overall_churn else 'lower'
print(f"\n➜ High-value churn rate is {direction} than overall by {abs(hv_churn - overall_churn):.1%}")


> **Business implication**: High-value customer churn represents direct revenue loss and warrants more retention investment than average customers.


## 5. Monthly Revenue Loss from Churn

Translate churn behavior into numbers you can report to management.


In [ ]:
churned = df[df["Churn"] == "Yes"]
retained = df[df["Churn"] == "No"]

monthly_loss = churned["MonthlyCharges"].sum()
avg_loss_per_customer = churned["MonthlyCharges"].mean()
avg_tenure_churned = churned["tenure"].mean()
avg_tenure_retained = retained["tenure"].mean()

print(f"Churned customers: {len(churned):,}")
print(f"Monthly revenue lost to churn: ${monthly_loss:,.0f}")
print(f"Average monthly charge per churned customer: ${avg_loss_per_customer:.2f}")
print(f"\nAverage tenure of churned customers: {avg_tenure_churned:.1f} months")
print(f"Average tenure of retained customers: {avg_tenure_retained:.1f} months")
print(f"\n➜ Retained customers stay {avg_tenure_retained - avg_tenure_churned:.1f} months longer than churned customers")


> **Business implication**: The tenure gap shows that intervening before customers churn can significantly increase customer lifetime value (LTV).


## 6. Which Service Combinations Best Retain Customers?

Identify low-churn service combinations to give the marketing team clear promotion targets.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Churn rate heatmap: Contract Type vs Internet Service
pivot = df.groupby(["Contract", "InternetService"])["Churn"].apply(
    lambda s: (s == "Yes").mean()
).unstack()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(pivot, annot=True, fmt=".1%", cmap="RdYlGn_r",
            linewidths=0.5, ax=axes[0])
axes[0].set_title("Churn Rate: Contract Type × Internet Service")
axes[0].set_xlabel("Internet Service")
axes[0].set_ylabel("Contract Type")

# Churn rate by payment method (bar chart)
pay_churn = df.groupby("PaymentMethod")["Churn"].apply(
    lambda s: (s == "Yes").mean()
).sort_values(ascending=True)

pay_churn.plot(kind="barh", ax=axes[1], color=["#2ecc71","#27ae60","#e67e22","#e74c3c"])
axes[1].set_title("Churn Rate by Payment Method")
axes[1].set_xlabel("Churn Rate")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

plt.tight_layout()
plt.savefig("churn_by_service.png", dpi=150, bbox_inches="tight")
plt.show()
print("Charts saved")

> **Business implication**: Contract length and payment method are the most direct behavioral signals. Marketing can prioritize promoting annual contracts + automatic payment combinations.


## 7. New Customer Danger Zone Analysis

Identify the tenure months with the highest churn concentration to help customer service set proactive outreach timing.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

# Group by tenure and calculate churn rate
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0, 3, 6, 12, 24, 48, 1000],
    labels=["1-3mo", "4-6mo", "7-12mo", "13-24mo", "25-48mo", "48mo+"],
    right=True
)

tenure_churn = df.groupby("tenure_group", observed=True)["Churn"].apply(
    lambda s: (s == "Yes").mean()
)

bars = ax.bar(tenure_churn.index, tenure_churn.values,
              color=["#e74c3c" if v > 0.3 else "#e67e22" if v > 0.2 else "#2ecc71"
                     for v in tenure_churn.values])

ax.axhline(y=(df["Churn"] == "Yes").mean(), color="gray",
           linestyle="--", label=f"Overall avg {(df['Churn'] == 'Yes').mean():.1%}")
ax.set_title("Churn Rate by Tenure Group")
ax.set_xlabel("Tenure")
ax.set_ylabel("Churn Rate")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.legend()

for bar, val in zip(bars, tenure_churn.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.1%}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("churn_by_tenure.png", dpi=150, bbox_inches="tight")
plt.show()

df.drop(columns=["tenure_group"], inplace=True)

> **Business implication**: The first 3 months are the highest-risk period. Recommend proactively contacting customers at months 1 and 3 with tailored offers or onboarding guidance.


## 8. Business Recommendations Summary

Three immediately actionable directions based on the analysis above.


In [ ]:
summary = {
    "Finding": [
        "Month-to-month customers churn at a far higher rate than one- or two-year contract customers",
        "Electronic check customers have the highest churn rate",
        "The first 3 months are the highest-risk period for new customers",
        "Fiber optic customers churn at a higher rate than DSL, suggesting a possible service quality gap",
    ],
    "Recommended Actions": [
        "[Contract Conversion] Offer annual contract discounts to month-to-month customers to drive upgrade",
        "[Payment Upgrade] Encourage electronic check customers to switch to automatic payment to reduce churn risk",
        "[New Customer Care] Establish proactive outreach at months 1 and 3 (call or email)",
        "[Fiber Quality] Investigate specific dissatisfaction drivers among Fiber customers and prioritize fixes",
    ]
}

print("=" * 50)
print("Customer Churn Analysis — Business Recommendations")
print("=" * 50)
for category, items in summary.items():
    print(f"\n[{category}]")
    for i, item in enumerate(items, 1):
        print(f"  {i}. {item}")
print("\n" + "=" * 50)


---

## 9. Project Scope

This project focuses on **business insights** — translating data into conclusions that stakeholders can act on:

- Churn rate analysis by segment (contract, payment method, tenure, service)
- Monthly revenue loss estimation
- New customer danger zone identification
- Actionable business recommendations

**Intentionally excluded**: machine learning models, prediction scores, A/B testing — these fall under more advanced data science work and are out of scope for this analysis.

> Data source: IBM Telco Customer Churn public dataset
